# Trimorious M1.4 — Verified Scientific Dataset Generation
**Pilot v0.1 | Separate candidates | 1,000 generated tasks | No training or publication**

This notebook implements an audit-first synthetic training-corpus pilot for the provisional Qwen3-4B-Instruct-2507 foundation. It does not use D07 as a teacher prompt, source of training tasks, or answer source. D07 questions are loaded only in the screening stage. Gold answers and historical model responses are never read by the generation engine.

Stages: configuration → trusted executable generators → independent checks → actual tool replay → private D07 screening → optional local teacher proposals → review queues → candidate-only export. Scientific text, interpretation, licensing and task diversity require independent human review. Even a computational pass is not a training approval. The templates are original synthetic exercises, not claims about real publications or real laboratory results. D03 retrieval licensing and D06 preference data are separate future work.

**Design disclosure:** the notebook author inspected D07 task-family patterns to deliberately choose different training task mechanisms; this means D07 is a known diagnostic benchmark, not an independent sealed confirmation test. The generation code does not ingest D07 questions or keys; a separate fresh benchmark is mandatory before final approval.

In [ ]:
%pip -q install numpy scipy sympy scikit-learn pandas tqdm

## 0 — Configuration
Upload the completed M1.3.2 FULL private audit ZIP into `/content`. CPU is sufficient for the default pilot. Optional teacher inference requires a GPU and a separately reviewed teacher checkpoint. No token is needed for local generation or screening. The default never downloads or calls a teacher. Keep all artifacts private until M1.5 review.

In [ ]:
import os, json, hashlib, zipfile, re, math, random, statistics, csv, platform, time
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
from fractions import Fraction
import numpy as np
import scipy
from scipy import stats, optimize
import sympy as sp
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd

ROOT = Path(os.environ.get('TRIMORIOUS_M14_ROOT','/content/trimorious_PRIVATE_m1_4'))
ROOT.mkdir(parents=True, exist_ok=True)
INPUT_ZIP = Path(os.environ.get('TRIMORIOUS_M14_INPUT','/content/trimorious_m1_3_2_PRIVATE_full_1c3d70c27cc3c9f8.zip'))
SEED = 184031
RUN_TEACHER = False
TEACHERS = []  # list of {"repo": ..., "revision": full 40-char commit SHA, "license_reviewed": True}
TEACHER_LIMIT = 30  # per teacher, proposals only; no automatic acceptance
TEACHER_MAX_NEW_TOKENS = 384
USE_DRIVE = False
ALLOW_TRAINING = False
ALLOW_PUBLISH = False
assert not ALLOW_TRAINING and not ALLOW_PUBLISH
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/trimorious_PRIVATE_m1_4')
    ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT = ROOT / RUN_ID
OUT.mkdir(exist_ok=False)
TARGETS = {'mathematical_reasoning':200,'scientific_reasoning':150,'agentic_tool_use':150,'statistical_reasoning':100,'research_methodology':100,'scientific_verification':100,'experimental_design':60,'literature_synthesis':50,'academic_articulation':50,'research_ethics_reliability':40}
PRIMARY_MODEL = {'repo':'Qwen/Qwen3-4B-Instruct-2507','revision':'cdbee75f17c01a7cc42f958dc650907174af0554','status':'provisional only'}

def canonical(x): return json.dumps(x, sort_keys=True, ensure_ascii=False, separators=(',',':'), allow_nan=False)
def digest(x): return hashlib.sha256(x if isinstance(x,bytes) else canonical(x).encode()).hexdigest()
def jsonl(path,rows):
    with path.open('w',encoding='utf-8') as f:
        for row in rows: f.write(canonical(row)+'\n')
def write_json(path,obj): path.write_text(json.dumps(obj,indent=2,ensure_ascii=False,allow_nan=False),encoding='utf-8')
def close(a,b): return math.isclose(float(a),float(b),rel_tol=1e-9,abs_tol=1e-13)
def num(x): return float(x)
def fmt(x): return format(float(x),'.12g')
print('Output:',OUT,'Targets:',sum(TARGETS.values()))

## 1 — Trusted numerical and tool execution library
Only registered operations on bounded, generator-owned arguments are executed. No Python expression, shell command, tool call or code from a teacher or input archive is executed. Each reference uses a different calculation path. These checks validate the stated numeric result for the supplied assumptions, not general scientific validity.

In [ ]:
def execute_tool(name, a):
    if name == 'trapezoid':
        x,y=a['x'],a['y']; return float(np.trapz(y,x)) if not hasattr(np,'trapezoid') else float(np.trapezoid(y,x))
    if name == 'weighted_average': return float(np.average(a['values'],weights=a['weights']))
    if name == 'percentile_linear': return float(np.percentile(a['values'],a['q'],method='linear'))
    if name == 'linear_fit_slope': return float(stats.linregress(a['x'],a['y']).slope)
    if name == 'root_bisection':
        return float(optimize.bisect(lambda x:x*x-a['target'],0,max(1,a['target']),xtol=1e-12))
    if name == 'binomial_tail': return float(stats.binom.sf(a['k']-1,a['n'],a['p']))
    raise ValueError('Unknown tool')

def tool_reference(name,a):
    if name=='trapezoid': return sum((a['x'][i+1]-a['x'][i])*(a['y'][i]+a['y'][i+1])/2 for i in range(len(a['x'])-1))
    if name=='weighted_average': return sum(v*w for v,w in zip(a['values'],a['weights']))/sum(a['weights'])
    if name=='percentile_linear':
        s=sorted(a['values']); h=(len(s)-1)*a['q']/100; i=int(h); return s[i]+(h-i)*(s[min(i+1,len(s)-1)]-s[i])
    if name=='linear_fit_slope':
        x,y=a['x'],a['y']; n=len(x); return (n*sum(u*v for u,v in zip(x,y))-sum(x)*sum(y))/(n*sum(u*u for u in x)-sum(x)**2)
    if name=='root_bisection': return math.sqrt(a['target'])
    if name=='binomial_tail': return sum(math.comb(a['n'],j)*a['p']**j*(1-a['p'])**(a['n']-j) for j in range(a['k'],a['n']+1))
    raise ValueError(name)

def pack(q,answer,reference=None,method=None,explanation='',rubric=None,evidence=None,trace=None,unit=None):
    return dict(question=q,answer=answer,reference=reference,check_method=method,explanation=explanation,rubric=rubric or [],evidence=evidence or [],tool_trace=trace,unit=unit)
def numeric(q,a,b,method,why,unit=None): return pack(q,num(a),num(b),method,why,unit=unit)
def narrative(q,answer,criteria,evidence=None): return pack(q,answer,rubric=criteria,evidence=evidence)

## 2 — Mathematics, physical science and statistics
Independent numerical checks use SymPy, exact rational arithmetic, explicit formulas or a separate SciPy/NumPy implementation. No benchmark task templates are reused intentionally. Parameterized instances are still correlated, so all exports carry template and scenario identifiers. Numeric answer explanations are not automatically certified merely because their answer matches.

In [ ]:
def math_task(t,r):
    a,b,c=[r.randint(2,45) for _ in range(3)]; x=sp.Symbol('x', positive=True)
    if t==0:
        q=f"A positive quantity satisfies y exp(y) = {a} exp({a}). Determine y and justify uniqueness on the positive real axis."
        return numeric(q,a,optimize.brentq(lambda y:y*math.exp(y)-a*math.exp(a),0,a+1),'monotone equation / Brent root','The function y exp(y) is strictly increasing for y>0; substitution gives the root.')
    if t==1:
        q=f"Evaluate the limit as h approaches zero of (sqrt({a}+{b}h)-sqrt({a}))/h."
        return numeric(q,b/(2*math.sqrt(a)),sp.limit((sp.sqrt(a+b*x)-sp.sqrt(a))/x,x,0),'rationalization / SymPy limit','Multiply numerator and denominator by the conjugate, then take the limit.')
    if t==2:
        q=f"A 2x2 symmetric matrix has diagonal entries {a+b}, {a+b} and off-diagonal entries {b}. Find its larger eigenvalue."
        return numeric(q,a+2*b,np.linalg.eigvalsh([[a+b,b],[b,a+b]])[-1],'eigenvector formula / eigvalsh','The vector (1,1) is an eigenvector with the larger eigenvalue.')
    if t==3:
        q=f"Find the probability of drawing two red objects without replacement from {a} red and {b} blue objects. Return a fraction or decimal."
        return numeric(q,Fraction(a*(a-1),(a+b)*(a+b-1)),math.comb(a,2)/math.comb(a+b,2),'sequential probability / combinations','Multiply the first and conditional second draw probabilities.')
    if t==4:
        q=f"Solve the initial-value problem dy/dt = -{a}y, y(0)={b}. Return y at t=0.1."
        return numeric(q,b*math.exp(-a*.1),sp.exp(-sp.Rational(a,10))*b,'separation of variables / symbolic expression','The exponential decay solution is y(t)=y(0) exp(-kt).')
    if t==5:
        q=f"For the complex number z={a}+{b}i, compute the imaginary part of 1/z."
        return numeric(q,-b/(a*a+b*b),(1/complex(a,b)).imag,'conjugate / complex arithmetic','Multiply by the complex conjugate and divide by squared magnitude.')
    if t==6:
        q=f"A function is expanded around x=0: exp({a}x). Find the coefficient of x^4 in its Taylor series."
        return numeric(q,Fraction(a**4,24),sp.series(sp.exp(a*x),x,0,5).removeO().coeff(x,4),'Taylor coefficient / symbolic series','The fourth-order term has coefficient a^4/4!.')
    if t==7:
        q=f"A rectangular grid graph has {a} rows and {b} columns of vertices; edges join immediate horizontal and vertical neighbors only. How many edges exist?"
        edges=sum((i+1<a)+(j+1<b) for i in range(a) for j in range(b))
        return numeric(q,a*(b-1)+b*(a-1),edges,'edge counting / enumeration','Count horizontal and vertical edges separately.')
    if t==8:
        q=f"Find the smallest positive integer divisible by {a}, {b}, and {c}."
        return numeric(q,math.lcm(a,b,c),sp.ilcm(a,b,c),'LCM / SymPy','Take the largest required prime power for each prime.')
    q=f"A right triangle has legs {a} and {b}. Find the radius of its inscribed circle."
    h=math.hypot(a,b)
    return numeric(q,a*b/(a+b+h),(a+b-h)/2,'area/semiperimeter / right-triangle identity','Use area divided by semiperimeter.')

def science_task(t,r):
    a,b,c=[r.randint(2,24) for _ in range(3)]
    if t==0:
        return numeric(f"An ideal resistor of {a} ohms carries {b/10:g} A for {c} s. Compute the electrical energy dissipated in joules.", (b/10)**2*a*c, (a*b/10)*(b/10)*c,'Joule heating / VIt','Use E=I²Rt, assuming constant resistance and current.','J')
    if t==1:
        return numeric(f"An ideal parallel-plate capacitor has capacitance {a} microfarads and voltage {b} V. Find stored energy in microjoules.",a*b*b/2,sp.Rational(a,2)*b*b,'capacitor energy / symbolic calculation','Use E=CV²/2; microfarads give microjoules.','µJ')
    if t==2:
        return numeric(f"Mix {a} kg of water at {20+b} °C with {c} kg at 10 °C, assuming no heat loss and identical specific heat. Compute equilibrium temperature in °C.",(a*(20+b)+c*10)/(a+c),float(sp.solve(a*(sp.Symbol('T')-20-b)+c*(sp.Symbol('T')-10),sp.Symbol('T'))[0]),'energy balance / symbolic solve','Net heat exchange is zero in the isolated system.','°C')
    if t==3:
        return numeric(f"A sound pulse returns from a reflector after {a/100:g} s. Assume speed {330+b} m/s and a straight round trip. Find one-way distance in meters.",(330+b)*a/200, (330+b)*(a/100)/2,'round-trip distance / unit conversion','The pulse covers twice the reflector distance.','m')
    if t==4:
        return numeric(f"A solution has hydrogen-ion activity 10^(-{a/4:g}). Under the given activity definition, calculate pH.",a/4,-math.log10(10**(-a/4)),'pH definition / logarithm','pH is minus the base-ten logarithm of hydrogen-ion activity.')
    if t==5:
        return numeric(f"A thin converging lens has focal length {a} cm and object distance {a+b} cm. Use the real-is-positive convention. Find real image distance in cm.",a*(a+b)/b,1/(1/a-1/(a+b)),'thin lens equation / reciprocal calculation','Apply 1/f=1/u+1/v.','cm')
    if t==6:
        return numeric(f"An ideal photon has wavelength {400+10*a} nm. Using c=299792458 m/s, calculate frequency in terahertz.",299792458/(400+10*a)/1000,299792458/((400+10*a)*1e-9)/1e12,'wave equation / SI conversion','Convert nanometers to meters and hertz to THz.','THz')
    if t==7:
        return numeric(f"A block of mass {a} kg occupies {b/100:g} m³. A fluid density is 1000 kg/m³. Find the ratio of block density to fluid density.",a/(10*b),(a/(b/100))/1000,'density ratio / SI division','The ratio is dimensionless.')
    if t==8:
        return numeric(f"Two ideal resistors of {a} and {b} ohms are connected in parallel. Compute equivalent resistance in ohms.",a*b/(a+b),1/(1/a+1/b),'parallel resistance / conductance addition','Conductances add for parallel resistors.','ohm')
    return numeric(f"A hypothetical turbine has shaft input {100*a} W and useful output {a*b} W. Compute fractional conversion efficiency.",b/100,(a*b)/(100*a),'energy efficiency / ratio','Output divided by input, assuming the stated powers refer to the same interval.')

def stats_task(t,r):
    a,b,c=[r.randint(2,18) for _ in range(3)]
    if t==0:
        q=f"A diagnostic study has {a} true positives, {b} false positives, and {c} false negatives. Compute positive predictive value."
        return numeric(q,Fraction(a,a+b),1/(1+b/a),'conditional probability / odds identity','PPV is TP/(TP+FP).')
    if t==1:
        v=[a,b,c,a+b,a+c,b+c]; q=f"Find the median absolute deviation about the median (unscaled) of {v}."
        return numeric(q,statistics.median([abs(x-statistics.median(v)) for x in v]),stats.median_abs_deviation(v),'median definition / SciPy','Use the median of absolute deviations; do not apply a normal consistency scale.')
    if t==2:
        q=f"A sample has {a} successes and {b} failures. Under a Beta(2,3) prior and binomial likelihood, calculate the posterior mean of the success probability."
        return numeric(q,(a+2)/(a+b+5),stats.beta.mean(a+2,b+3),'conjugacy / SciPy beta mean','Update the beta shape parameters with the counts.')
    if t==3:
        q=f"Independent Poisson counts have expected totals {a} and {b}. What is the probability both counts are zero?"
        return numeric(q,math.exp(-(a+b)),stats.poisson.pmf(0,a)*stats.poisson.pmf(0,b),'Poisson additivity / independent PMFs','Independence implies multiplication of zero-count probabilities.')
    if t==4:
        q=f"In a trial, group A has {a} events and {b} non-events; group B has {c} events and {b} non-events. Compute the odds ratio A versus B."
        return numeric(q,a/c,(a*b)/(b*c),'odds ratio / cross-product','Take the ratio of event odds.')
    if t==5:
        q=f"A normal random variable has mean {a} and standard deviation {b}. Compute P(X ≤ {a+b})."
        return numeric(q,(1+math.erf(1/math.sqrt(2)))/2,stats.norm.cdf(a+b,loc=a,scale=b),'normal CDF / erf','Standardize to z=1.')
    if t==6:
        q=f"A classifier produces probabilities [{a/20:g}, {b/20:g}, {c/20:g}] for outcomes [1,0,1]. Compute the Brier score."
        p=[a/20,b/20,c/20]; y=[1,0,1]
        return numeric(q,sum((u-v)**2 for u,v in zip(p,y))/3,np.mean((np.array(p)-np.array(y))**2),'squared probability loss / NumPy','Average squared probability errors.')
    if t==7:
        q=f"A study tests {a} hypotheses. Bonferroni familywise alpha is 0.05. Compute the per-test rejection threshold."
        return numeric(q,.05/a,Fraction(1,20*a),'Bonferroni / exact fraction','Divide the familywise threshold by the number of tests.')
    if t==8:
        q=f"In a simple reliability model, {a} independent components each work with probability 0.9. A series system requires all to work. Find system reliability."
        return numeric(q,.9**a,stats.binom.pmf(a,a,.9),'independence / binomial probability','All components must succeed.')
    q=f"A paired experiment has differences [{a},{-b},{c},0,{a-b}]. Find the root mean squared difference, not the standard deviation."
    v=[a,-b,c,0,a-b]
    return numeric(q,math.sqrt(sum(x*x for x in v)/len(v)),np.linalg.norm(v)/math.sqrt(len(v)),'RMS definition / vector norm','Divide the squared norm by the number of differences before taking the square root.')

## 3 — Real tool execution
Six tool families; each candidate records the exact arguments, execution output, independent reference, runtime and a second replay. Tool observations are generated by code, not invented by a teacher. The neutral trace is not yet a Qwen-native chat template; conversion to a training format requires an explicit integration test in M1.5/M2.0.

In [ ]:
def agent_task(t,r):
    a,b,c=[r.randint(2,25) for _ in range(3)]
    if t==0:
        name='trapezoid'; args={'x':[0,1,3,6],'y':[a,b,c,a+b]}; goal='Estimate the area under the sampled curve using the composite trapezoid rule.'
    elif t==1:
        name='weighted_average'; args={'values':[a,b,c],'weights':[2,3,5]}; goal='Combine three calibration readings using the stated weights.'
    elif t==2:
        name='percentile_linear'; args={'values':[a,b,c,a+b,b+c,a+c],'q':r.choice([25,40,60,75])}; goal='Calculate the requested percentile using linear interpolation.'
    elif t==3:
        name='linear_fit_slope'; args={'x':[0,1,3,5],'y':[a,a+b,a+2*b+c,a+4*b]}; goal='Estimate the ordinary least-squares slope with an intercept.'
    elif t==4:
        name='root_bisection'; args={'target':a*b+c}; goal='Find the positive real square root by a bracketed numerical root solver.'
    else:
        name='binomial_tail'; args={'n':a+5,'k':r.randint(2,a+4),'p':b/30}; goal='Find the probability of at least k successes in n independent Bernoulli trials.'
    q=f"{goal} Use the registered {name} tool with arguments {canonical(args)}. Report the observed result without claiming an experiment was performed."
    start=time.perf_counter(); result=execute_tool(name,args); elapsed=time.perf_counter()-start
    ref=tool_reference(name,args); replay=execute_tool(name,args)
    trace={'tool':name,'arguments':args,'observation':result,'independent_reference':ref,'replay_result':replay,'replay_pass':close(result,replay) and close(result,ref),'execution_seconds':elapsed,'execution_kind':'local_registered_function'}
    return pack(q,num(result),num(ref),'registered tool / independent calculation',f'The executed {name} returned {fmt(result)}.',trace=trace)

## 4 — Evidence-grounded research instructions
All evidence cards below are explicitly fictional. They provide the entire evidence for each task, including the quantities and limitations stated. Reference responses are authored rule-based drafts, not verified literature reviews. They remain in a human-review queue. No fabricated citations, DOIs, or real research results are inserted.

In [ ]:
DOMAINS=['battery cycling','river sediment monitoring','plant germination','ceramic durability','acoustic insulation','solar coating tests','robotic positioning','soil moisture sensing','membrane permeability','optical alignment','composite fatigue','microfluidic mixing','thermal insulation','satellite image segmentation','enzyme stability','wind tunnel testing','marine sensor drift','polymer aging','spectrometer calibration','grain storage']
# Each entry: a distinct research design/evidence pattern, a scenario, a reference response, a rubric.
CARDS = {
'research_methodology':[
('missingness','Sensor outages occur primarily during high readings.','Missingness may depend on the unobserved outcome. Characterize outages, report missingness patterns and conduct sensitivity analyses under plausible missing-not-at-random assumptions.','Identify informative missingness; avoid assuming MCAR; propose sensitivity analysis.'),
('cluster','An intervention is assigned to entire facilities, with multiple observations per facility.','The assignment unit is the facility. Account for within-facility dependence and use a cluster-level or hierarchical analysis; report the number of independent facilities.','Identify assignment unit; account for clustering; avoid pseudoreplication.'),
('measurement','A reference standard and a new method disagree more at high values.','Inspect paired differences against magnitude and assess proportional bias. Correlation alone does not establish agreement.','Distinguish agreement from correlation; assess magnitude-dependent bias.'),
('model_selection','Several candidate models are compared using repeated tuning on the same development set.','Use nested resampling for model selection and performance estimation, or a genuinely untouched external test set. Record the tuning process.','Separate tuning and estimation; specify nested or untouched evaluation.'),
('estimand','Some assigned units stop the intervention early but outcomes are still measured.','Define the estimand before analysis. An assignment-policy effect can retain units in their assigned groups; other estimands need explicit assumptions and handling of deviations.','Define estimand; distinguish assignment from adherence; document assumptions.')],
'scientific_verification':[
('seed','A code package documents its algorithm but not random seeds or library versions.','Attempt reconstruction using recorded inputs, pin dependencies and record seeds; distinguish a similar result from exact reproduction.','Name missing run conditions; distinguish exact and approximate reproduction.'),
('uncertainty','Two instruments give values with known calibration uncertainty but their errors may share a common reference.','Do not assume independence. Model shared calibration covariance and propagate uncertainty; compare only after checking the measurement definitions.','Account for covariance; align measurands.'),
('boundary','A fitted equation agrees with observations inside its calibration interval, but is applied far outside it.','Label the prediction as extrapolation and test it against independent observations in the new range before claiming validity.','Recognize extrapolation; request independent validation.'),
('conservation','A numerical simulation yields a state that violates its explicitly stated conserved quantity beyond tolerance.','Flag a numerical or model consistency failure. Check discretization, solver tolerance, boundary fluxes and conservation assumptions before trusting output.','Identify inconsistency; propose diagnostics; avoid unverified conclusions.'),
('version','Two analysis reports use a dataset with the same title but different preprocessing scripts.','Compare file and preprocessing hashes, align inclusion rules and rerun from an identified version; title equality is not identity.','Verify hashes and transformations; require traceable rerun.')],
'experimental_design':[
('factorial','Two controllable settings may interact. Resources allow testing all combinations of two levels per setting.','Use a replicated two-by-two factorial design and estimate main effects and interaction, with run order randomized where feasible.','Include all four cells; replication; interaction.'),
('dose','A response may have a threshold and a plateau. Only one active level has been examined.','Choose several justified levels spanning the expected transition plus a reference level; define outcomes and assess the response curve with uncertainty.','Multiple levels; reference; uncertainty; no unjustified linearity.'),
('drift','Calibration drifts slowly throughout a day. The intervention can be switched between short runs.','Interleave and randomize conditions within time blocks, with reference checks throughout the day. Model residual time drift if necessary.','Counter temporal drift; checks; randomization within blocks.')],
'research_ethics_reliability':[
('authorship','An automated system proposes manuscript text and analyses, while humans direct and verify the work.','Disclose the system use according to applicable journal policy, document human responsibility and verify every claim. Do not attribute accountable authorship to software.','Disclosure; human responsibility; verification.'),
('dual_role','An evaluator has a financial interest in a vendor whose system is being assessed.','Disclose the conflict, use independent assessment and prespecified criteria, and retain an auditable decision record.','Disclose conflict; independent evaluation; transparency.'),
('consent_scope','A proposed secondary analysis is outside the consent and data-use scope of the original collection.','Do not proceed until the appropriate ethics and governance review confirms a lawful and permitted basis. Minimize data and restrict access.','Respect consent scope; obtain review; minimize data.'),
('image','An image adjustment is applied selectively to a region to strengthen an apparent scientific signal.','Preserve the original image, document transformations and use consistent justified processing. Do not alter an image in a way that misrepresents evidence.','Original preservation; consistent processing; no misleading alteration.')],
}

def research_task(f,t,r,i):
    domain=DOMAINS[i%len(DOMAINS)]
    key,scenario,answer,criteria=CARDS[f][t]
    # Contextual framing is varied, but the template pattern is explicitly retained for diversity and split audits.
    q=f"Fictional research planning memo: {domain}. Scenario: {scenario} Give a defensible next action and explain its rationale without inventing observations."
    return narrative(q,answer,criteria.split('; '),[{'id':'scenario','kind':'fictional_case','text':scenario}])

def literature_task(t,r,i):
    domain=DOMAINS[i%len(DOMAINS)]; a,b=r.randint(12,80),r.randint(20,100)
    if t==0:
        A=f'A: {a} samples; effect estimated in a controlled laboratory range only.'; B=f'B: {b} samples; field study covers a wider range but uses an uncalibrated instrument.'
        ans='[A] supports a conclusion only within its laboratory range. [B] covers a wider range but measurement validity is uncertain. The studies do not establish a transferable effect without calibration and independent validation.'; rub=['Use both cards','Distinguish coverage from measurement validity','Avoid inferring an effect size']
    elif t==1:
        A=f'A: {a} units; outcome defined as change from initial value.'; B=f'B: {b} units; outcome defined as final absolute value.'
        ans='[A] and [B] use different outcome definitions. Direct pooling is not justified without a common estimand and suitable data. Report the results separately until harmonization is possible.'; rub=['Recognize incompatible outcomes','Do not pool automatically','Cite cards']
    elif t==2:
        A=f'A: {a} observations; fitted a linear model within observed conditions.'; B=f'B: {b} observations; reports curvature over a different interval.'
        ans='[A] describes a local linear approximation, while [B] describes curvature in a different interval. These claims may coexist; compare overlapping ranges and model assumptions before declaring contradiction.'; rub=['Distinguish ranges','Avoid declaring contradiction without evidence','Cite both']
    elif t==3:
        A=f'A: {a} runs; reports median latency only.'; B=f'B: {b} runs; reports 95th-percentile latency only.'
        ans='[A] characterizes typical latency and [B] characterizes the upper tail. The metrics are not interchangeable; a shared distribution or matched quantiles are needed for quantitative comparison.'; rub=['Different quantiles','No direct ranking','Specify needed evidence']
    else:
        A=f'A: {a} records; released raw data but no transformation log.'; B=f'B: {b} records; released transformation scripts but no raw inputs.'
        ans='[A] and [B] each provide partial reproducibility evidence. Neither card establishes a complete rerunnable pipeline. Obtain both identified inputs and executable transformations before claiming reproduction.'; rub=['Identify complementary gaps','No verified reproduction claim','Cite both']
    return narrative(f"Fictional evidence cards for {domain}: {A} {B} Write a three-sentence synthesis citing [A] and [B]. Do not invent publications or results.",ans,rub,[{'id':'A','kind':'fictional_card','text':A},{'id':'B','kind':'fictional_card','text':B}])

def academic_task(t,r,i):
    domain=DOMAINS[i%len(DOMAINS)]; a,b=r.randint(12,90),r.randint(2,30)
    if t==0:
        case=f'A simulation of {domain} used {a} parameter settings; no physical measurements were collected.'; ans=f'The simulation of {domain} explored {a} parameter settings. Its outputs characterize the specified computational model; physical validity has not been established.'; rub=['Label simulation','Preserve number','Avoid physical validation claim']
    elif t==1:
        case=f'A calibration series for {domain} included {a} runs; {b%a} runs lacked a recorded reference value.'; ans=f'The calibration series comprised {a} runs, of which {b%a} lacked a recorded reference value. The missing reference values limit the assessment of measurement agreement.'; rub=['Preserve counts','Describe missing reference','Do not invent direction of bias']
    elif t==2:
        case=f'A computational comparison for {domain} reports median latency {b} ms on one device; hardware transfer was not tested.'; ans=f'The recorded median latency was {b} ms on the evaluated device. Performance on other hardware remains untested.'; rub=['Preserve unit','Restrict claim to device','No invented speedup']
    elif t==3:
        case=f'A protocol for {domain} proposes {a} planned runs; none have been performed.'; ans=f'The proposed protocol specifies {a} planned runs. No experimental findings are available because the runs have not yet been performed.'; rub=['Distinguish planned from performed','No fabricated results','Preserve number']
    else:
        case=f'A model for {domain} was fitted to {a} observations; prediction intervals were not calculated.'; ans=f'The model was fitted to {a} observations. Prediction uncertainty was not quantified, limiting interpretation of its predictions.'; rub=['Preserve fitting sample','Note uncertainty gap','No invented interval']
    return narrative(f"Write a cautious two-sentence academic statement using only this fictional record: {case}",ans,rub,[{'id':'record','kind':'fictional_case','text':case}])

## 5 — Generate and verify all 1,000 pilot candidates
Duplicate prompts are rejected and regenerated. Narrative templates are counted explicitly; only numeric and tool results can pass machine verification. Every example, including a passed numerical result, remains unapproved for training. For diversity, M1.5 should add independently authored research cases and new task mechanisms rather than only scaling the parameter ranges.

In [ ]:
TEMPLATE_COUNTS={'mathematical_reasoning':10,'scientific_reasoning':10,'agentic_tool_use':6,'statistical_reasoning':10,'research_methodology':5,'scientific_verification':5,'experimental_design':3,'literature_synthesis':5,'academic_articulation':5,'research_ethics_reliability':4}
DATASETS={'mathematical_reasoning':'D02','scientific_reasoning':'D01','agentic_tool_use':'D04','statistical_reasoning':'D05','research_methodology':'D01','scientific_verification':'D09','experimental_design':'D01','literature_synthesis':'D01','academic_articulation':'D01','research_ethics_reliability':'D09'}
GENERATORS={'mathematical_reasoning':math_task,'scientific_reasoning':science_task,'agentic_tool_use':agent_task,'statistical_reasoning':stats_task}
rng=random.Random(SEED); records=[]; seen=set(); failures=[]
for family,target in TARGETS.items():
    ntemplates=TEMPLATE_COUNTS[family]
    for i in range(target):
        t=i%ntemplates
        for attempt in range(500):
            if family in GENERATORS: p=GENERATORS[family](t,rng)
            elif family=='literature_synthesis': p=literature_task(t,rng,i+attempt)
            elif family=='academic_articulation': p=academic_task(t,rng,i+attempt)
            else: p=research_task(family,t,rng,i//ntemplates+attempt)
            qhash=digest(p['question'].strip().casefold())
            if qhash not in seen: break
        else: raise RuntimeError(f'Insufficient unique prompts: {family}/{t}')
        seen.add(qhash)
        numeric_check=p['reference'] is not None
        passed=bool(close(p['answer'],p['reference'])) if numeric_check else None
        if p['tool_trace']: passed=passed and p['tool_trace']['replay_pass']
        if passed is False: failures.append({'family':family,'template':t,'question_hash':qhash})
        response=(p['explanation']+' Result: '+fmt(p['answer'])+((' '+p['unit']) if p['unit'] else '')+'.') if numeric_check else p['answer']
        rec={'id':f'M14-{len(records)+1:05d}','dataset':DATASETS[family],'family':family,'template_id':f'{family}_{t:02d}','prompt_sha256':qhash,'prompt':p['question'],'reference_response':response,'numeric_answer':p['answer'] if numeric_check else None,'independent_reference':p['reference'],'unit':p['unit'],'verification':{'status':'numeric_and_replay_pass' if passed and p['tool_trace'] else 'numeric_pass' if passed else 'numeric_fail' if passed is False else 'human_rubric_required','method':p['check_method'],'passed':passed},'tool_trace':p['tool_trace'],'evidence':p['evidence'],'rubric':p['rubric'],'provenance':{'kind':'original_synthetic_template','generator_version':'M1.4-pilot-v0.1','seed':SEED,'template_index':t,'source_license':'original generated material; redistribution review pending','external_source':None},'human_decision':'pending','training_approved':False}
        # Content fingerprint excludes wall-clock execution time.
        stable={k:v for k,v in rec.items() if k!='tool_trace'}
        stable['tool_trace']={k:v for k,v in rec['tool_trace'].items() if k!='execution_seconds'} if rec['tool_trace'] else None
        rec['record_sha256']=digest(stable)
        records.append(rec)
assert len(records)==1000 and len(seen)==1000
assert Counter(x['family'] for x in records)==Counter(TARGETS)
assert len({x['template_id'] for x in records})==sum(TEMPLATE_COUNTS.values())
assert not failures, failures
print(pd.DataFrame([{'family':r['family'],'status':r['verification']['status']} for r in records]).value_counts().to_string())
print('Distinct templates:',len({x['template_id'] for x in records}))

## 6 — Mandatory D07 privacy and leakage gate
The full M1.3.2 archive must match the frozen D07 question-file hash and benchmark ID. No extraction to disk is performed; no gold key or historical response is opened. D07 is never included in teacher prompts, candidate exports, or training records. Character and word TF-IDF cosine similarities plus a number-masked exact check are conservative indicators, not proof of no semantic overlap. Every flagged candidate is quarantined; an independent reviewer must still assess shared mechanisms and paraphrases. The threshold is fixed before inspecting results.

In [ ]:
EXPECTED_BENCHMARK='47694f1b2f4dbd44e8f5b6e58962ad07af5c2a844e375249e74d785de0b51283'
EXPECTED_QUESTIONS='e724980d46b644f47952d3ab87789e94a32cb0b79835c4721ad3c7616c311c7e'
assert INPUT_ZIP.is_file(), f'Upload the completed private M1.3.2 archive: {INPUT_ZIP}'
with zipfile.ZipFile(INPUT_ZIP) as z:
    assert z.testzip() is None, 'Corrupt input archive'
    manifest=json.loads(z.read('PRIVATE_DO_NOT_TRAIN/manifest.json'))
    qbytes=z.read('PRIVATE_DO_NOT_TRAIN/D07_questions.jsonl')
    assert manifest['benchmark_sha256']==EXPECTED_BENCHMARK and digest(qbytes)==EXPECTED_QUESTIONS
    assert manifest['file_hashes']['D07_questions.jsonl']==EXPECTED_QUESTIONS and manifest['training_use'] is False
    prior=[json.loads(l)['question'] for l in qbytes.splitlines() if l.strip()]
assert len(prior)==1000 and len(set(prior))==1000

def norm(s): return re.sub(r'\s+',' ',s.casefold()).strip()
def mask(s): return re.sub(r'\b\d+(?:\.\d+)?\b','NUM',norm(s))
prior_exact=set(map(norm,prior)); prior_masked=set(map(mask,prior))
# Fit only for screening; vectors and benchmark text are never exported.
CHAR_THRESHOLD=.78; WORD_THRESHOLD=.72

def similarity_max(candidate,prior,analyzer,ngram_range):
    vec=TfidfVectorizer(analyzer=analyzer,ngram_range=ngram_range,sublinear_tf=True,norm='l2',dtype=np.float32)
    m=vec.fit_transform([*prior,*candidate]); a=m[:len(prior)]; b=m[len(prior):]
    return np.array((b@a.T).max(axis=1).toarray()).ravel()
questions=[r['prompt'] for r in records]
char_scores=similarity_max([mask(q) for q in questions],[mask(q) for q in prior],'char',(3,5))
word_scores=similarity_max([mask(q) for q in questions],[mask(q) for q in prior],'word',(1,2))
for r,c,w in zip(records,char_scores,word_scores):
    exact=norm(r['prompt']) in prior_exact; masked=mask(r['prompt']) in prior_masked
    flagged=bool(exact or masked or c>=CHAR_THRESHOLD or w>=WORD_THRESHOLD)
    r['leakage_screen']={'exact_match':exact,'number_masked_match':masked,'char_similarity':float(c),'word_similarity':float(w),'flagged':flagged,'semantic_review':'pending'}
    r['disposition']='quarantine_leakage' if flagged else 'quarantine_verification_failure' if r['verification']['passed'] is False else 'candidate_numeric_checked_review_pending' if r['verification']['passed'] is True else 'candidate_rubric_review_pending'
# Do not retain the benchmark text in the subsequent teacher/generation scope.
del prior,qbytes,prior_exact,prior_masked,manifest
leakage_report={'benchmark_sha256':EXPECTED_BENCHMARK,'questions_sha256':EXPECTED_QUESTIONS,'benchmark_questions_screened':1000,'candidate_questions_screened':len(records),'char_threshold':CHAR_THRESHOLD,'word_threshold':WORD_THRESHOLD,'flagged_count':sum(r['leakage_screen']['flagged'] for r in records),'max_char_similarity':float(max(char_scores)),'max_word_similarity':float(max(word_scores)),'scope':'question lexical and number-masked screening only; semantic/pretraining contamination unknown','gold_key_read':False}
print(leakage_report)

# Freeze template groups before any optional teacher inference.
eligible=[r for r in records if not r['disposition'].startswith('quarantine')]
quarantine=[r for r in records if r['disposition'].startswith('quarantine')]
val_templates=set()
for f in TARGETS:
    templates=sorted({r['template_id'] for r in eligible if r['family']==f})
    if len(templates)>1:
        chosen=random.Random(SEED+sum(map(ord,f))).sample(templates,max(1,round(.15*len(templates))))
        val_templates.update(chosen)
for r in eligible: r['candidate_split']='validation_candidate' if r['template_id'] in val_templates else 'development_candidate'
for r in quarantine: r['candidate_split']='quarantined'
dev=[r for r in eligible if r['candidate_split']=='development_candidate']; val=[r for r in eligible if r['candidate_split']=='validation_candidate']
assert set(r['template_id'] for r in dev).isdisjoint(r['template_id'] for r in val)
assert len(dev)+len(val)+len(quarantine)==1000
assert not any(r['training_approved'] for r in records)
assert all(r['verification']['passed'] is not False and not r['leakage_screen']['flagged'] for r in eligible)


## 7 — Optional teacher-model orchestration (disabled)
The pilot is useful without GPU inference. To request additional local model proposals, supply one or more licensed, independently reviewed Hugging Face teacher checkpoints pinned to immutable commit hashes. The provisional Qwen3 foundation is not automatically a superior teacher. This optional runner sends only non-quarantined development-candidate prompts (never the validation split) and never sends D07 content, golds or tool observations. A teacher cannot execute tools here or fabricate accepted observations. Numeric proposals must supply an unambiguous JSON numeric answer and match the independently computed reference; prose, units and rationale require review. Narrative proposals always require rubric review. No teacher output replaces the reference or enters the candidate corpus automatically. Teacher model loading has not been GPU-tested.

In [ ]:
teacher_proposals=[]; teacher_errors=[]
if RUN_TEACHER:
    assert TEACHERS, 'Configure reviewed local teachers first'
    import subprocess,sys,gc
    subprocess.check_call([sys.executable,'-m','pip','-q','install','transformers>=4.51,<5','accelerate','bitsandbytes','huggingface_hub'])
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from huggingface_hub import HfApi
    assert torch.cuda.is_available(), 'Teacher runner requires CUDA GPU'
    teacher_eligible=[r for r in dev if not r['tool_trace']]
    # Deterministic family round-robin for broad coverage.
    buckets={f:[r for r in teacher_eligible if r['family']==f] for f in TARGETS}
    selected=[]
    for j in range(max(map(len,buckets.values()))):
        for b in buckets.values():
            if j<len(b) and len(selected)<TEACHER_LIMIT: selected.append(b[j])
        if len(selected)>=TEACHER_LIMIT: break
    for teacher in TEACHERS:
        model=tokenizer=None
        try:
            assert teacher.get('license_reviewed') is True and re.fullmatch(r'[0-9a-f]{40}',teacher['revision'])
            info=HfApi().model_info(teacher['repo'],revision=teacher['revision'])
            assert info.sha==teacher['revision'] and any(s.rfilename.endswith('.safetensors') for s in info.siblings)
            tokenizer=AutoTokenizer.from_pretrained(teacher['repo'],revision=info.sha,trust_remote_code=False)
            model=AutoModelForCausalLM.from_pretrained(teacher['repo'],revision=info.sha,trust_remote_code=False,use_safetensors=True,device_map='auto',quantization_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.float16)).eval()
            for r in selected:
                try:
                    directive='Return one JSON object with numeric field answer and string field explanation.' if r['numeric_answer'] is not None else 'Return a concise evidence-grounded response. Do not invent data or citations.'
                    messages=[{'role':'system','content':directive},{'role':'user','content':r['prompt']}]
                    ids=tokenizer.apply_chat_template(messages,add_generation_prompt=True,return_tensors='pt').to(model.device)
                    with torch.inference_mode(): result=model.generate(ids,attention_mask=torch.ones_like(ids),max_new_tokens=TEACHER_MAX_NEW_TOKENS,do_sample=False,pad_token_id=tokenizer.eos_token_id)
                    answer=tokenizer.decode(result[0,ids.shape[1]:],skip_special_tokens=True)
                    status='human_rubric_required'
                    if r['numeric_answer'] is not None:
                        try:
                            v=json.loads(answer.strip())['answer']; status='numeric_match_review_pending' if type(v) in (int,float) and math.isfinite(v) and close(v,r['independent_reference']) else 'numeric_mismatch'
                        except Exception: status='unparseable'
                    teacher_proposals.append({'candidate_id':r['id'],'prompt_sha256':r['prompt_sha256'],'teacher_repo':teacher['repo'],'teacher_revision':info.sha,'response':answer,'status':status,'training_approved':False})
                except Exception as e: teacher_errors.append({'teacher':teacher['repo'],'candidate_id':r['id'],'error':str(e)[:500]})
        except Exception as e: teacher_errors.append({'teacher':teacher.get('repo'),'error':str(e)[:500]})
        finally:
            del model,tokenizer; gc.collect(); torch.cuda.empty_cache()
print('Teacher proposals:',len(teacher_proposals),'Errors:',len(teacher_errors),'Enabled:',RUN_TEACHER)

## 8 — Candidate split, review and export
A deterministic holdout groups entire templates within each capability. It is a proposed development/validation split, not a training-ready corpus or a fresh sealed confirmation benchmark. D04 traces are exported as neutral structured records; D10 is a tool registry, not 150 additional independent examples. All candidates carry explicit approval flags set to false. The audit contains no benchmark questions, keys, or benchmark responses. Teacher proposals are separate and always unapproved.

In [ ]:
# Export in isolated directories; no filenames implying release-approved training data.
for directory in ['candidate_datasets','quarantine','reports','teacher_review','tool_registry']:(OUT/directory).mkdir()
for dataset in sorted(set(DATASETS.values())):
    for split,items in [('development_candidate',dev),('validation_candidate',val)]:
        jsonl(OUT/'candidate_datasets'/f'{dataset}_{split}.jsonl',[r for r in items if r['dataset']==dataset])
jsonl(OUT/'quarantine'/'quarantined.jsonl',quarantine)
jsonl(OUT/'teacher_review'/'unapproved_teacher_proposals.jsonl',teacher_proposals)
write_json(OUT/'teacher_review'/'teacher_errors.json',teacher_errors)
write_json(OUT/'tool_registry'/'D10_registry.json',{'status':'pilot only','operations':['trapezoid','weighted_average','percentile_linear','linear_fit_slope','root_bisection','binomial_tail'],'execution_policy':'registered bounded generator-owned arguments only','tool_trace_count':sum(r['tool_trace'] is not None for r in records),'native_chat_template_validated':False})
review=pd.DataFrame([{'id':r['id'],'dataset':r['dataset'],'family':r['family'],'template_id':r['template_id'],'prompt':r['prompt'],'reference_response':r['reference_response'],'evidence':canonical(r['evidence']),'rubric':canonical(r['rubric']),'verification':r['verification']['status'],'disposition':r['disposition'],'candidate_split':r['candidate_split'],'reviewer':'','decision':'pending','notes':''} for r in records])
review.to_csv(OUT/'reports'/'human_review_queue.csv',index=False)
ledger=[{'id':r['id'],'record_sha256':r['record_sha256'],'prompt_sha256':r['prompt_sha256'],'dataset':r['dataset'],'family':r['family'],'template_id':r['template_id'],'verification':r['verification']['status'],'disposition':r['disposition'],'candidate_split':r['candidate_split'],'training_approved':False} for r in records]
jsonl(OUT/'reports'/'record_ledger.jsonl',ledger)
counts=Counter(r['disposition'] for r in records)
# Identical answer text and shared templates are reported, not hidden by distinct prompt hashes.
diversity=[]
for family in TARGETS:
    rows=[r for r in records if r['family']==family]
    answer_groups=Counter(norm(r['reference_response']) for r in rows)
    template_groups=Counter(r['template_id'] for r in rows)
    diversity.append({'family':family,'records':len(rows),'unique_prompts':len({r['prompt_sha256'] for r in rows}),'unique_reference_responses':len(answer_groups),'largest_identical_response_group':max(answer_groups.values()),'distinct_templates':len(template_groups),'largest_template_group':max(template_groups.values()),'review_status':'pending_semantic_and_pedagogical_review'})
pd.DataFrame(diversity).to_csv(OUT/'reports'/'diversity_audit.csv',index=False)
assert close(1,1) and not close(1,2) and not close(1e-10,2e-10)
assert all(r['candidate_split']=='development_candidate' for r in dev)
assert all(p['candidate_id'] in {r['id'] for r in dev} for p in teacher_proposals)
summary={'total_generated':len(records),'unique_prompts':len(seen),'distinct_templates':len({r['template_id'] for r in records}),'by_family':dict(Counter(r['family'] for r in records)),'by_dataset':dict(Counter(r['dataset'] for r in records)),'numerically_checked':sum(r['verification']['passed'] is True for r in records),'numerical_failures':len(failures),'actual_tool_replays_passed':sum(bool(r['tool_trace'] and r['tool_trace']['replay_pass']) for r in records),'rubric_required':sum(r['verification']['passed'] is None for r in records),'dispositions':dict(counts),'development_candidates':len(dev),'validation_candidates':len(val),'quarantined':len(quarantine),'approved_for_training':0,'teacher_proposals':len(teacher_proposals),'teacher_enabled':RUN_TEACHER,'frozen_D07_hash':EXPECTED_BENCHMARK,'primary_model':PRIMARY_MODEL,'training_performed':False,'publishing_performed':False,'seed':SEED,'runtime':{'python':platform.python_version(),'numpy':np.__version__,'scipy':scipy.__version__,'sympy':sp.__version__,'scikit_learn':sklearn.__version__}}
write_json(OUT/'reports'/'summary.json',summary);write_json(OUT/'reports'/'leakage_report.json',leakage_report)
review.groupby(['family','disposition']).size().rename('count').reset_index().to_csv(OUT/'reports'/'quality_scorecard.csv',index=False)
write_json(OUT/'reports'/'split_manifest.json',{'validation_templates':sorted(val_templates),'group_overlap':0,'split_status':'provisional; scenario and semantic review pending'})
write_json(OUT/'reports'/'release_gates.json',{'numerical_checks':'passed','lexical_gate':'executed; flagged rows quarantined','human_review':'pending','scientific_expert_review':'pending','semantic_leakage_review':'pending','teacher_quality_review':'pending or not run','license_review':'pending','fresh_confirmation_benchmark':'pending','D03_provenance':'separate pending','Qwen_native_format_test':'pending','training_approved':False,'release_approved':False})
(OUT/'README.md').write_text('# Trimorious M1.4 private pilot\nGenerated synthetic candidate material only. All examples require human review; no approved training examples or public release. D07 is excluded. D03, D06 and D08 are deferred. D10 contains a registry of the six implemented tool functions. Template counts are not independent task counts. The optional teacher outputs are quarantined as proposals and are not merged. Numerical checks do not establish the scientific or pedagogical quality of prose. No real experiments were performed. License and source review are required before redistribution.\n',encoding='utf-8')
file_hashes={str(p.relative_to(OUT)):digest(p.read_bytes()) for p in sorted(OUT.rglob('*')) if p.is_file()}
write_json(OUT/'archive_manifest.json',file_hashes)
ZIP=ROOT/f'trimorious_m1_4_PRIVATE_pilot_{RUN_ID}.zip'
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob('*')):
        if p.is_file(): z.write(p,str(p.relative_to(OUT)))
with zipfile.ZipFile(ZIP) as z:
    assert z.testzip() is None
    assert all(digest(z.read(n))==h for n,h in file_hashes.items())
    assert not any('D07_questions' in n or 'D07_gold' in n for n in z.namelist())
print(json.dumps(summary,indent=2));print('Private audit ZIP:',ZIP)

## 9 — M1.4 pilot exit checklist
- Verify the generated candidate counts, source/record hashes, numerical reference checks and replay results.
- Inspect the review queue, including explanations, rubrics, assumptions and synthetic evidence; commission domain experts for genuine scientific task diversity.
- Review number-masked and semantic overlap with D07 privately. Create a new sealed independent confirmation benchmark before tuning or selecting checkpoints.
- Confirm licensing and Qwen chat-template rendering; do not ingest D03 or D07 into SFT.
- Review optional teacher proposals independently. No teacher answer is approved automatically.
- Proceed to M1.5 only after human review and a corrected candidate dataset manifest. Training and publishing are separate milestones.

In [ ]:
# Download manually from Colab's Files sidebar, or set DOWNLOAD_ZIP=True.
DOWNLOAD_ZIP=False
if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(ZIP))